# "THE PRICE IS RIGHT" 
最终大项目这周你要做的，就是基于从亚马逊爬下来的数据，构建一个模型，根据商品描述来预测它的价格。
# 课程安排
**第一天**：数据整理

**第二天**：数据预处理

**第三天**：评估、基线模型和传统机器学习

**第四天**：深度学习和大语言模型

**第五天**：微调前沿模型

## 第五天：用百炼微调 Qwen

现在我们要把前面准备好的商品定价数据，整理成对话微调数据集，并用阿里云百炼支持的 Qwen 模型创建一个我们自己的价格预测模型。

托管微调平台变化很快：OpenAI 新组织不能再创建新的自助微调任务，硅基流动也已经停止新建微调服务。所以这里切换到仍提供模型调优接口的阿里云百炼。核心流程不变：准备 `messages` JSONL，上传训练/验证文件，创建 SFT 任务，最后调用微调后的模型评估效果。


In [74]:
# imports

import os
import re
import json
import requests
from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from pricer.items  import Item
from pricer.evaluator import evaluate


In [75]:
# environment

LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [76]:
username = "iloveawscn"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 800,000 training items, 10,000 validation items, 10,000 test items


In [80]:
BAILIAN_API_BASE = "https://dashscope.aliyuncs.com/api/v1"
BAILIAN_COMPATIBLE_API_BASE = "https://dashscope.aliyuncs.com/compatible-mode/v1"
BAILIAN_API_KEY_ENV = "DASHSCOPE_API_KEY"

# 演示建议先用较小的 Qwen Instruct 模型，速度和成本更稳。
BAILIAN_FINE_TUNE_BASE_MODEL = "qwen2.5-7b-instruct"

dashscope_api_key = os.getenv(BAILIAN_API_KEY_ENV, "").strip()

bailian = OpenAI(
    api_key=dashscope_api_key or "missing-dashscope-api-key",
    base_url=BAILIAN_COMPATIBLE_API_BASE,
)

if dashscope_api_key:
    print("DashScope API key loaded")
else:
    print("请先在 .env 中设置 DASHSCOPE_API_KEY，然后重新运行这个单元格")

BAILIAN_FINE_TUNE_BASE_MODEL


DashScope API key loaded


'qwen2.5-7b-instruct'

关于数据量，托管微调通常建议先用一个小规模的数据集，快速验证格式、流程和效果。

百炼的 SFT 数据量下限可以覆盖我们这里的课堂演示规模，所以这里先用 100 条训练样本和 50 条验证样本。这个规模足够演示完整流程，等待时间和成本也更容易控制。


In [81]:
# 先用小规模数据完成录课演示：100 条训练样本 + 50 条验证样本
# 如果你要认真追求效果，可以把样本数量逐步放大，再重新创建微调任务。

fine_tune_train = train[:100]
fine_tune_validation = val[:50]


In [82]:
len(fine_tune_train)

100

第一步，准备数据。把商品数据转成对话微调需要的 JSONL 格式。

每一行都是一个训练样本，里面有一组 `messages`：用户给商品描述，assistant 给出标准价格答案。这个结构是 SFT 最常见的数据格式，也可以用于百炼的模型调优流程。


In [83]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

In [84]:
messages_for(fine_tune_train[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single‑piece oil‑rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4" minimum center‑to‑center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation.'},
 {'role': 'assistant', 'content': '$64.30'}]

In [85]:
# Convert the items into a list of json objects - a "jsonl" string
# Each row represents a message in the form:
# {"messages" : [{"role": "system", "content": "You estimate prices...


def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str +'}\n'
    return result.strip()

In [86]:
print(make_jsonl(train[:3]))

{"messages": [{"role": "user", "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single\u2011piece oil\u2011rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4\" minimum center\u2011to\u2011center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation."}, {"role": "assistant", "content": "$64.30"}]}
{"messages": [{"role": "user", "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Mini Electric Air Duster Fan  \nCategory: Electronics  \nBrand: Kica  \nDescription: Ultra\u2011compact 86,000\u202fRPM electric air duster with 11\u202fm/s wind speed for precise cleaning and inflation.  \nDetails: Powered by a 9.99\u202fWh motor, adjustable in four speed levels, it uses three 

In [87]:
# Convert the items into jsonl and write them to a file

from pathlib import Path

JSONL_DIR = Path("jsonl")
JSONL_DIR.mkdir(exist_ok=True)


def write_jsonl(items, filename):
    with open(filename, "w", encoding="utf-8") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl + "\n")


In [53]:
write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")

In [54]:
write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation.jsonl")

In [88]:
# 本地校验 JSONL 文件：百炼上传的就是这两个文件

train_jsonl_path = JSONL_DIR / "fine_tune_train.jsonl"
validation_jsonl_path = JSONL_DIR / "fine_tune_validation.jsonl"

for jsonl_path in [train_jsonl_path, validation_jsonl_path]:
    with open(jsonl_path, encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
    print(f"{jsonl_path}: {len(rows)} rows")
    print(rows[0])
    print()


jsonl/fine_tune_train.jsonl: 100 rows
{'messages': [{'role': 'user', 'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single‑piece oil‑rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4" minimum center‑to‑center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation.'}, {'role': 'assistant', 'content': '$64.30'}]}

jsonl/fine_tune_validation.jsonl: 50 rows
{'messages': [{'role': 'user', 'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: SAFUEL 10,000mAh Magnetic Portable Charger  \nCategory: Electronics  \nBrand: SAFUEL  \nDescription: A compact 10,000mAh power bank that snaps magnetically to MagSafe‑enabled iPhones for fast wireless charging.  \nDetails: Offers 20W US

现在我们把刚才生成的两个文件上传到阿里云百炼：

- 训练文件：`jsonl/fine_tune_train.jsonl`
- 验证文件：`jsonl/fine_tune_validation.jsonl`
- 基座模型：建议先使用 `qwen2.5-7b-instruct`
- 调优方法：SFT
- 训练轮数：先用 `1` 轮

百炼调优任务 API 文档：<https://help.aliyun.com/zh/model-studio/create-fine-tuning-job-api>

注意：创建微调任务可能产生费用。下面的上传文件可以直接运行；创建训练任务那一步默认有开关，确认要开始训练后再打开。


In [ ]:
def require_dashscope_api_key():
    if not dashscope_api_key:
        raise EnvironmentError("请先在 .env 中设置 DASHSCOPE_API_KEY，然后重新运行环境和百炼配置单元格")
    return dashscope_api_key


def bailian_headers():
    return {"Authorization": f"Bearer {require_dashscope_api_key()}"}


def show_bailian_response(response):
    try:
        data = response.json()
    except ValueError:
        print(response.text)
        response.raise_for_status()
        return None

    print(json.dumps(data, ensure_ascii=False, indent=2))

    if response.status_code == 403 and data.get("code") == "AccessDenied" and "sfm_modelunits_public_cn" in data.get("message", ""):
        raise RuntimeError(
            "百炼模型部署服务尚未开通：sfm_modelunits_public_cn。"
            "请先在百炼/阿里云控制台开通模型部署/模型单元服务，再重新运行部署单元格。"
        )

    response.raise_for_status()
    return data


def walk_json(value):
    if isinstance(value, dict):
        yield value
        for child in value.values():
            yield from walk_json(child)
    elif isinstance(value, list):
        for child in value:
            yield from walk_json(child)


def extract_id(data, *names):
    for obj in walk_json(data):
        for name in names:
            value = obj.get(name)
            if value:
                return value

    raise KeyError(f"无法从响应中提取 ID，候选字段：{names}")


In [90]:
def upload_bailian_fine_tune_file(jsonl_path):
    with open(jsonl_path, "rb") as f:
        response = requests.post(
            f"{BAILIAN_API_BASE}/files",
            headers=bailian_headers(),
            data={"purpose": "fine-tune"},
            files={"files": (jsonl_path.name, f, "application/jsonl")},
            timeout=120,
        )

    data = show_bailian_response(response)
    file_id = extract_id(data, "file_id", "id")
    return file_id, data


train_file_id, train_file_response = upload_bailian_fine_tune_file(train_jsonl_path)
validation_file_id, validation_file_response = upload_bailian_fine_tune_file(validation_jsonl_path)

print("Training file id:", train_file_id)
print("Validation file id:", validation_file_id)


{
  "data": {
    "uploaded_files": [
      {
        "name": "fine_tune_train.jsonl",
        "file_id": "e867a085-0379-43de-808f-34fee1066774"
      }
    ],
    "failed_uploads": []
  },
  "request_id": "f02ce355-1d0d-9c33-ba76-743c8473782c"
}
{
  "data": {
    "uploaded_files": [
      {
        "name": "fine_tune_validation.jsonl",
        "file_id": "5d966ed2-7319-4ee1-b725-a2f0de716b7c"
      }
    ],
    "failed_uploads": []
  },
  "request_id": "d796ca2a-4114-9188-bf41-ce4a0d87d505"
}
Training file id: e867a085-0379-43de-808f-34fee1066774
Validation file id: 5d966ed2-7319-4ee1-b725-a2f0de716b7c


## 第二步：微调时间到

接下来创建百炼模型调优任务。这里的关键参数是：

- `model`：从哪个 Qwen 基座模型开始训练
- `training_file_ids`：刚上传的训练文件 ID
- `validation_file_ids`：刚上传的验证文件 ID
- `training_type`：这里使用 `sft`
- `hyper_parameters.n_epochs`：训练轮数，演示先用 1

为了避免误触发付费训练，下面的单元格默认不会创建任务。确认要开始训练时，把 `RUN_BAILIAN_FINE_TUNE` 改成 `True` 再运行。


In [58]:
RUN_BAILIAN_FINE_TUNE = True

fine_tune_payload = {
    "model": BAILIAN_FINE_TUNE_BASE_MODEL,
    "training_file_ids": [train_file_id],
    "validation_file_ids": [validation_file_id],
    "training_type": "sft",
    "hyper_parameters": {
        "n_epochs": 1,
    },
}

print(json.dumps(fine_tune_payload, ensure_ascii=False, indent=2))

if RUN_BAILIAN_FINE_TUNE:
    response = requests.post(
        f"{BAILIAN_API_BASE}/fine-tunes",
        headers={**bailian_headers(), "Content-Type": "application/json"},
        json=fine_tune_payload,
        timeout=120,
    )
    fine_tune_job = show_bailian_response(response)
else:
    fine_tune_job = None
    print("RUN_BAILIAN_FINE_TUNE=False，尚未创建训练任务。确认要训练时改成 True 后重新运行。")


{
  "model": "qwen2.5-7b-instruct",
  "training_file_ids": [
    "eed4f9b6-fd29-4365-a64a-9bd5b8631ffc"
  ],
  "validation_file_ids": [
    "56f5c953-19a6-40bc-898a-6734630c2f97"
  ],
  "training_type": "sft",
  "hyper_parameters": {
    "n_epochs": 1
  }
}
{
  "request_id": "ebe9ada7-870b-978f-9ab9-ec387500c046",
  "output": {
    "job_id": "ft-202606091633-9852",
    "job_name": "ft-202606091633-9852",
    "status": "PENDING",
    "finetuned_output": "qwen2.5-7b-instruct-ft-202606091633-9852",
    "model": "qwen2.5-7b-instruct",
    "base_model": "qwen2.5-7b-instruct",
    "training_file_ids": [
      "eed4f9b6-fd29-4365-a64a-9bd5b8631ffc"
    ],
    "validation_file_ids": [
      "56f5c953-19a6-40bc-898a-6734630c2f97"
    ],
    "training_datasets": [],
    "validation_datasets": [],
    "hyper_parameters": {
      "n_epochs": 1,
      "batch_size": "16",
      "learning_rate": "1e-5"
    },
    "training_type": "sft",
    "create_time": "2026-06-09 16:33:55",
    "workspace_id": "w

In [59]:
# 如果上一个单元格已经创建任务，这里会自动提取 job id。
# 也可以从百炼控制台复制任务 ID，写到环境变量 BAILIAN_FINE_TUNE_JOB_ID。

BAILIAN_FINE_TUNE_JOB_ID = os.getenv("BAILIAN_FINE_TUNE_JOB_ID", "").strip()

if fine_tune_job:
    BAILIAN_FINE_TUNE_JOB_ID = extract_id(fine_tune_job, "job_id", "fine_tune_job_id", "id")

BAILIAN_FINE_TUNE_JOB_ID


'ft-202606091633-9852'

In [91]:
def retrieve_bailian_fine_tune_job(job_id):
    if not job_id:
        raise ValueError("请先创建百炼调优任务，或者把任务 ID 填入 BAILIAN_FINE_TUNE_JOB_ID")

    response = requests.get(
        f"{BAILIAN_API_BASE}/fine-tunes/{job_id}",
        headers=bailian_headers(),
        timeout=60,
    )
    return show_bailian_response(response)


# 任务创建后可以重复运行这一格，查看排队、训练、完成状态。
if BAILIAN_FINE_TUNE_JOB_ID:
    job_status = retrieve_bailian_fine_tune_job(BAILIAN_FINE_TUNE_JOB_ID)
else:
    print("还没有 BAILIAN_FINE_TUNE_JOB_ID，等创建任务后再运行这一格。")


{
  "request_id": "d9c6ef99-af38-97d8-a08f-47f75f8caa49",
  "output": {
    "job_id": "ft-202606091633-9852",
    "job_name": "ft-202606091633-9852",
    "status": "SUCCEEDED",
    "finetuned_output": "qwen2.5-7b-instruct-ft-202606091633-9852",
    "model": "qwen2.5-7b-instruct",
    "base_model": "qwen2.5-7b-instruct",
    "training_file_ids": [
      "eed4f9b6-fd29-4365-a64a-9bd5b8631ffc"
    ],
    "validation_file_ids": [
      "56f5c953-19a6-40bc-898a-6734630c2f97"
    ],
    "training_datasets": [],
    "validation_datasets": [],
    "hyper_parameters": {
      "n_epochs": 1,
      "batch_size": "16",
      "learning_rate": "1e-5"
    },
    "training_type": "sft",
    "create_time": "2026-06-09 16:33:55",
    "workspace_id": "ws-eww6pp3v2uvbcbok",
    "user_identity": "1609760246064938",
    "modifier": "1609760246064938",
    "creator": "1609760246064938",
    "end_time": "2026-06-09 17:17:30",
    "group": "llm",
    "usage": 13696,
    "model_name": "ft-202606091633-9852",
  

## 第三步：部署并测试微调后的模型

注意：百炼里“调优任务完成 / 模型发布成功”还不等于可以直接用 `chat.completions.create()` 调用。调优完成后会得到一个 `finetuned_output`，它是可部署模型；还需要再创建一个部署实例。部署状态为 `RUNNING` 后，响应里的 `deployed_model` 才是推理时要传入 `model=` 的名称。

下面先列举当前账号可部署的自定义模型，确认我们的调优产物在列表里。


In [92]:
def get_job_output(job_response):
    if isinstance(job_response, dict) and isinstance(job_response.get("output"), dict):
        return job_response["output"]
    return {}


job_output = get_job_output(job_status if "job_status" in globals() else fine_tune_job)
BAILIAN_FINE_TUNED_OUTPUT = os.getenv("BAILIAN_FINE_TUNED_OUTPUT", "").strip()

if not BAILIAN_FINE_TUNED_OUTPUT:
    BAILIAN_FINE_TUNED_OUTPUT = job_output.get("finetuned_output", "")

BAILIAN_FINE_TUNED_OUTPUT


'qwen2.5-7b-instruct-ft-202606091633-9852'

In [99]:
def list_bailian_deployable_models(model_source="custom"):
    response = requests.get(
        f"{BAILIAN_API_BASE}/deployments/models",
        headers={**bailian_headers(), "Content-Type": "application/json"},
        params={"page_no": 1, "page_size": 100, "version": "v1.0", "model_source": model_source},
        timeout=60,
    )
    return show_bailian_response(response)


deployable_models = list_bailian_deployable_models("custom")

matching_models = [
    model for model in deployable_models.get("output", {}).get("models", [])
    if model.get("model_name") == BAILIAN_FINE_TUNED_OUTPUT
]

if matching_models:
    print("Found deployable model:", matching_models[0]["model_name"])
else:
    print("没有在可部署模型列表里找到：", BAILIAN_FINE_TUNED_OUTPUT)
    print("如果任务刚完成，请稍等一会儿再重新运行这一格。")


{
  "output": {
    "models": [
      {
        "model_name": "qwen2.5-7b-instruct-ft-202606091633-9852",
        "plans": [
          {
            "templates": [
              {
                "template_name": "单机部署",
                "charge_type": "pre_paid",
                "roles": {
                  "unified": {
                    "model_unit_spec": "MU5",
                    "capacity_unit_per_instance": 1
                  }
                },
                "template_source": "SYSTEM",
                "template_type": "COUPLED",
                "template_id": "dps-20260513200132-s2bbmc",
                "template_desc": "高性价比部署方案，在保证合理推理性能的同时优化单位成本，适合预算敏感但需稳定服务的中低负载场景。",
                "deploy_spec": "dps-20260513200132-s2bbmc"
              },
              {
                "template_name": "单机部署",
                "charge_type": "post_paid",
                "roles": {
                  "unified": {
                    "model_unit_spec": "MU5",
                    "capac

现在创建部署实例。部署可能产生费用，所以这里也设置了显式开关。确认要部署时，把 `RUN_BAILIAN_DEPLOYMENT` 改成 `True` 再运行。

注意：不同调优模型支持的部署方案不同。我们不再硬编码 `lora`，而是从上一步 `/deployments/models` 返回的 `plans/templates` 里自动选择一个可用的后付费 `mu` 模板，并据此填写 `deploy_spec`、`billing_method` 和 `capacity`。

如果这里返回 `Service not activated : sfm_modelunits_public_cn`，说明当前阿里云账号还没有开通百炼模型部署/模型单元服务。这个不是代码问题，需要先到控制台开通部署服务后再继续。


In [108]:
RUN_BAILIAN_DEPLOYMENT = True

BAILIAN_DEPLOYMENT_NAME = os.getenv("BAILIAN_DEPLOYMENT_NAME", "qwen-price-pricer-202606091633").strip()


def choose_bailian_deployment_template(model_info):
    for plan in model_info.get("plans", []):
        if plan.get("plan") != "mu":
            continue

        templates = plan.get("templates", [])
        post_paid_templates = [t for t in templates if t.get("charge_type") == "post_paid"]
        candidate_templates = post_paid_templates or templates

        if candidate_templates:
            template = candidate_templates[0]
            capacity = template.get("roles", {}).get("unified", {}).get("capacity_unit_per_instance", 1)
            return plan.get("plan"), template, capacity

    raise ValueError("这个调优模型没有可用的 mu 部署模板，请回到可部署模型列表查看支持的部署方案")


if not matching_models:
    raise ValueError("没有找到可部署的调优模型，请先重新运行上一个单元格确认模型已出现在可部署列表中")

selected_plan, selected_template, selected_capacity = choose_bailian_deployment_template(matching_models[0])
selected_deploy_spec = selected_template.get("deploy_spec") or selected_template.get("template_id")

if not selected_deploy_spec:
    raise ValueError("可部署模板中没有 deploy_spec/template_id，无法创建部署")

deployment_payload = {
    "model_name": BAILIAN_FINE_TUNED_OUTPUT,
    "plan": selected_plan,
    "deploy_spec": selected_deploy_spec,
    "billing_method": "POST_PAY",
    "capacity": selected_capacity,
    "name": BAILIAN_DEPLOYMENT_NAME,
}

print("Selected deployment template:")
print(json.dumps(selected_template, ensure_ascii=False, indent=2))
print("Deployment payload:")
print(json.dumps(deployment_payload, ensure_ascii=False, indent=2))

if RUN_BAILIAN_DEPLOYMENT:
    response = requests.post(
        f"{BAILIAN_API_BASE}/deployments",
        headers={**bailian_headers(), "Content-Type": "application/json"},
        json=deployment_payload,
        timeout=120,
    )
    deployment_job = show_bailian_response(response)
else:
    deployment_job = None
    print("RUN_BAILIAN_DEPLOYMENT=False，尚未创建部署。确认要部署时改成 True 后重新运行。")


Selected deployment template:
{
  "template_name": "单机部署",
  "charge_type": "post_paid",
  "roles": {
    "unified": {
      "model_unit_spec": "MU5",
      "capacity_unit_per_instance": 1
    }
  },
  "template_source": "SYSTEM",
  "template_type": "COUPLED",
  "template_id": "dps-20260513200132-s2bbmc",
  "template_desc": "高性价比部署方案，在保证合理推理性能的同时优化单位成本，适合预算敏感但需稳定服务的中低负载场景。",
  "deploy_spec": "dps-20260513200132-s2bbmc"
}
Deployment payload:
{
  "model_name": "qwen2.5-7b-instruct-ft-202606091633-9852",
  "plan": "mu",
  "deploy_spec": "dps-20260513200132-s2bbmc",
  "billing_method": "POST_PAY",
  "capacity": 1,
  "name": "qwen-price-pricer-202606091633"
}
{
  "request_id": "0e372a4d-042d-93fe-b974-ca4625d32a0e",
  "output": {
    "name": "qwen-price-pricer-202606091633",
    "deployed_model": "qwen2.5-7b-instruct-b639e0f9a6fa",
    "gmt_create": "2026-06-10T16:05:25.484",
    "gmt_modified": "2026-06-10T16:05:25.484",
    "status": "PENDING",
    "model_name": "qwen2.5-7b-instruct-ft-20

In [109]:
# 如果已经创建部署，这里会自动提取 deployed_model。
# 也可以从百炼控制台复制部署实例 ID，写入环境变量 BAILIAN_DEPLOYED_MODEL。

BAILIAN_DEPLOYED_MODEL = os.getenv("BAILIAN_DEPLOYED_MODEL", "").strip()

if deployment_job:
    BAILIAN_DEPLOYED_MODEL = extract_id(deployment_job, "deployed_model", "deployment_id", "id")

BAILIAN_DEPLOYED_MODEL


'qwen2.5-7b-instruct-b639e0f9a6fa'

In [110]:
def retrieve_bailian_deployment(deployed_model):
    if not deployed_model:
        raise ValueError("请先创建部署，或者把部署实例 ID 填入 BAILIAN_DEPLOYED_MODEL")

    response = requests.get(
        f"{BAILIAN_API_BASE}/deployments/{deployed_model}",
        headers={**bailian_headers(), "Content-Type": "application/json"},
        timeout=60,
    )
    return show_bailian_response(response)


if BAILIAN_DEPLOYED_MODEL:
    deployment_status = retrieve_bailian_deployment(BAILIAN_DEPLOYED_MODEL)
else:
    print("还没有 BAILIAN_DEPLOYED_MODEL。创建部署后再运行这一格，等状态变成 RUNNING。")


{
  "request_id": "aff043b5-0bec-9f05-81cd-0f0a93183c37",
  "output": {
    "name": "qwen-price-pricer-202606091633",
    "deployed_model": "qwen2.5-7b-instruct-b639e0f9a6fa",
    "gmt_create": "2026-06-10T16:05:25",
    "gmt_modified": "2026-06-10T16:12:00",
    "status": "RUNNING",
    "model_name": "qwen2.5-7b-instruct-ft-202606091633-9852",
    "base_model": "qwen2.5-7b-instruct",
    "base_capacity": 1,
    "capacity": 1,
    "ready_capacity": 1,
    "workspace_id": "ws-eww6pp3v2uvbcbok",
    "charge_type": "post_paid",
    "creator": "1609760246064938",
    "modifier": "1609760246064938",
    "plan": "mu",
    "model_unit_spec": "dps-20260513200132-s2bbmc",
    "quantization_precision": "fp8",
    "qpm_limit": 600000,
    "tpm_limit": 100000000,
    "mu_capacity": {
      "unified": {
        "base_capacity": 1,
        "model_unit_spec": "MU5",
        "ready_capacity": 1,
        "capacity": 1
      }
    }
  }
}


In [111]:
# 部署状态为 RUNNING 后，把 deployed_model 填在这里。
# 注意：这里不是 ft-... 任务 ID，也不是 finetuned_output，而是部署实例 ID。
# 也可以写入环境变量 BAILIAN_DEPLOYED_MODEL。

if not BAILIAN_DEPLOYED_MODEL:
    BAILIAN_DEPLOYED_MODEL = "qwen2.5-7b-instruct-b639e0f9a6fa"

BAILIAN_DEPLOYED_MODEL


'qwen2.5-7b-instruct-b639e0f9a6fa'

In [113]:
# The prompt

def test_messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation{item.summary}"
    return [
        {"role": "user", "content": message},
    ]


test_messages_for(test[0])


[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanationTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [114]:
# The inference function


def qwen_fine_tuned(item):
    if BAILIAN_DEPLOYED_MODEL.startswith("replace-with"):
        raise ValueError("请先把 BAILIAN_DEPLOYED_MODEL 替换成百炼部署完成后的 deployed_model")

    response = bailian.chat.completions.create(
        model=BAILIAN_DEPLOYED_MODEL,
        messages=test_messages_for(item),
        temperature=0,
        max_tokens=7,
    )
    return response.choices[0].message.content


print(test[0].price)
print(qwen_fine_tuned(test[0]))


219.0
$160.00


In [ ]:
# 录课时建议先用小样本确认链路跑通；如果平台响应稳定，再逐步增大 size。
evaluate(qwen_fine_tuned, test, size=100, workers=1)

# 记录你实际跑出来的结果，例如：
# 训练样本 100 / 验证样本 50 / qwen2.5-7b-instruct / 1 epoch / MAE: ...


![Qwen Fine-Tuned Results](fin.png)